# **動画からの特徴量抽出**

このNotebookでは、NumPy 2系と現在のMediaPipe Tasks APIを使い、MP4動画から左右42点の手指ランドマークを抽出する。旧 `mp.solutions.hands` の再現ではなく、Hand Landmarkerが出力する画像座標を学習用特徴量へ変換する。


## 0. Colabの実行環境とHand Landmarkerモデルを準備する

次のセルでは、NumPy 2系を維持したまま現在のMediaPipeをインストールする。Hand Landmarkerは推論モデルを別途必要とするため、Google公式の `hand_landmarker.task` を `/content/models` へダウンロードする。その後、元動画と生成結果を読み書きするGoogle Driveをマウントする。


In [ ]:
# NumPy 2系とMediaPipe Tasks APIを使用します。
%pip install -q --upgrade "numpy>=2,<3" "mediapipe>=0.10.30" opencv-python-headless pandas scipy matplotlib tqdm

from importlib.metadata import version
from pathlib import Path
from urllib.request import urlretrieve

print('NumPy:', version('numpy'))
print('MediaPipe:', version('mediapipe'))

HAND_LANDMARKER_MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/hand_landmarker/'
    'hand_landmarker/float16/latest/hand_landmarker.task'
)
HAND_LANDMARKER_MODEL = Path('/content/models/hand_landmarker.task')
HAND_LANDMARKER_MODEL.parent.mkdir(parents=True, exist_ok=True)
if not HAND_LANDMARKER_MODEL.exists():
    urlretrieve(HAND_LANDMARKER_MODEL_URL, HAND_LANDMARKER_MODEL)
print('Hand Landmarkerモデル:', HAND_LANDMARKER_MODEL)

from google.colab import drive
drive.mount('/content/drive')


## 1. 前処理コードを取得する

`GITHUB_OWNER` をリポジトリ所有者へ変更します。次のセルは `main` ブランチを `/content/SimpleSignRecog` にcloneします。clone済みの場合も `main` へ切り替え、以後のセルが前処理モジュールをimportできるよう、作業ディレクトリをリポジトリ直下へ変更します。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# 前処理モジュールを含むブランチを明示して、同じコードを取得します。
GITHUB_OWNER = 'akio-kobayashi'
REPOSITORY_URL = f'https://github.com/{GITHUB_OWNER}/SimpleSignRecog.git'
REPOSITORY_BRANCH = 'main'
REPOSITORY_DIR = Path('/content/SimpleSignRecog')

if not REPOSITORY_DIR.exists():
    subprocess.run(
        [
            'git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch',
            REPOSITORY_URL, str(REPOSITORY_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(
        ['git', '-C', str(REPOSITORY_DIR), 'checkout', REPOSITORY_BRANCH],
        check=True,
    )

os.chdir(REPOSITORY_DIR)
print('作業ディレクトリ:', Path.cwd())
print('使用ブランチ:', REPOSITORY_BRANCH)


## **2. 入出力フォルダの確認**

次のセルでは，被験者ID，元の動画の場所，抽出結果と左右補正結果の保存先を設定する．
入力は `{subject_01,subject_02,subject_03}/1`～`{subject_01,subject_02,subject_03}/20` の各クラスにMP4を置く構成である．
入力ディレクトリがない場合や動画が0本のクラスがある場合は処理を停止する

In [ ]:
SUBJECT_ID = 'subject_03'#@param{type:'string'}  # 匿名化した識別子
INPUT_DIR = Path('/content/drive/Shareddrives/研究（山下）/SignData01.dra/'+SUBJECT_ID)
OUTPUT_ROOT = Path('/content/drive/Shareddrives/研究（山下）/SimpleSignRecogExp')
RAW_OUTPUT_DIR = OUTPUT_ROOT / 'data' / SUBJECT_ID
CORRECTED_OUTPUT_DIR = OUTPUT_ROOT / 'corrected' / SUBJECT_ID

assert INPUT_DIR.is_dir(), f'入力ディレクトリがありません: {INPUT_DIR}'

counts = {class_id: len(list((INPUT_DIR / str(class_id)).glob('*.mp4')))
          for class_id in range(1, 21)}
print('クラスごとの動画数:', counts)
print('合計:', sum(counts.values()))
assert all(counts.values()), '動画が0本のクラスがあります'

## **3. 1本の動画から126次元の特徴量を抽出する**

次のセルでは最初に見つかったMP4を読み、Hand LandmarkerをVIDEOモードで実行する。各フレームには動画のFPSから計算した時刻を渡すため、検出器は前フレームの結果を追跡に利用できる。

各手について21点の画像座標 `(x, y, z)` を使う。`x` と `y` は画像幅・高さで正規化された座標、`z` は手首を基準とする奥行きである。左手を列 `0:63`、右手を列 `63:126` に格納し、未検出部分は `NaN` とする。

左右ラベルが欠けた場合だけ補完規則を使い、その動画を `inferred` と記録する。続いて左右割当補正、時間方向の線形補間、手首・手の大きさ・掌の向きを基準にした正規化を適用し、各段階の欠損数を表示する。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing.landmark_extraction import extract_landmarks_from_video
from src.correct_inferred_data import determine_and_correct_handedness_globally
from src.preprocessing.missing_data import interpolate_missing_data
from src.preprocessing.normalization import canonical_normalize_landmarks

sample_video = next(INPUT_DIR.rglob('*.mp4'))
print('サンプル動画:', sample_video)

landmarks, had_inference, total_frames = extract_landmarks_from_video(
    sample_video,
    model_asset_path=HAND_LANDMARKER_MODEL,
)

# MediaPipe が左右を推定できなかったフレームを含む場合は，
# 動画全体の手首位置から左右の割当をそろえる
if had_inference:
    corrected_landmarks = determine_and_correct_handedness_globally(landmarks)
else:
    corrected_landmarks = landmarks.copy()

hands_swapped = not np.array_equal(
    landmarks, corrected_landmarks, equal_nan=True
)
interpolated = interpolate_missing_data(corrected_landmarks)
normalized = canonical_normalize_landmarks(interpolated)

print('形状:', landmarks.shape)
print('動画の総フレーム数:', total_frames)
print('左右推定を含む:', had_inference)
print('左右を入れ替えた:', hands_swapped)
print('抽出直後の欠損数:', np.isnan(landmarks).sum())
print('補間後の欠損数:', np.isnan(interpolated).sum())


## **4. 欠損補間と正規化の結果を表示する**

次のセルでは，右手首の`x`座標である列63を使って，左右補正後の欠損を含む系列と線形補間後の系列を同じグラフに描く．
また，補間後と正規化後の有限値の最小値・最大値を表示し，座標変換による値域の変化を確かめる．


In [ ]:
# 右手首 x 座標（列63）を例に，左右補正後の欠損と補間結果を比較する
frame = np.arange(len(corrected_landmarks))
plt.figure(figsize=(12, 4))
plt.plot(frame, corrected_landmarks[:, 63], 'o-', markersize=3, label='左右補正後（補間前）')
plt.plot(frame, interpolated[:, 63], '-', linewidth=2, label='線形補間後')
plt.xlabel('フレーム番号')
plt.ylabel('右手首 x 座標')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('正規化前の有限値範囲:', np.nanmin(interpolated), np.nanmax(interpolated))
print('正規化後の有限値範囲:', np.nanmin(normalized), np.nanmax(normalized))


## **5. 全ての動画からランドマークを抽出する**

次のセルは、クラス1～20の全MP4を同じHand LandmarkerモデルとVIDEOモードで処理する。抽出した `(フレーム数, 126)` のfloat32配列を `processed_data/<クラス>/<動画名>.npz` の `landmarks` キーへ保存する。NPZパス、クラス番号、元動画、左右ラベル補完の有無、フレーム数は `metadata.csv` に記録する。


In [ ]:
from src.process_videos import create_dataset

metadata = create_dataset(
    input_root_dir=INPUT_DIR,
    output_base_dir=RAW_OUTPUT_DIR,
    model_asset_path=HAND_LANDMARKER_MODEL,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
display(metadata.head())
print('作成したサンプル数:', len(metadata))

## **6. 動画単位で左右割当を補正する**

次のセルは，作成した `metadata.csv` を読み，`quality_flag` が `inferred` のNPZだけ左右割当を判定する．最初の検出フレームで画面左側を左手，画面右側を右手とみなし，逆なら全フレームの左右63列を交換する．
`clean` のNPZはそのままコピーし，補正済みNPZと新しい `metadata.csv` を `CORRECTED_OUTPUT_DIR` に保存する．


In [ ]:
# inferred の動画のみ，動画全体を見て左右割当を補正する
subprocess.run(
    [
        sys.executable, 'src/correct_inferred_data.py',
        '--input_base_dir', str(RAW_OUTPUT_DIR),
        '--output_base_dir', str(CORRECTED_OUTPUT_DIR),
    ],
    check=True,
)


## **7. 学習時に行う前処理**

保存したNPZは抽出直後のランドマークである．
学習時の `SignDataset` は，NPZ読込後に線形補間，学習用データ拡張，設定された正規化，Savitzky–Golay平滑化を順に適用する．
その後，位置・速度・加速度・指先間距離・正規化前の手首速度を結合して1フレーム392次元にし，残る `NaN` と無限大を0へ置換する．
バッチ作成時には，動画ごとに異なる系列長を0埋めでそろえる．
